Pool Distance
==============
Ths notebook is a little bit more complicated. It loads our median income census data for NYC tracts and
locations of public pools from NYC Open Data. It then merges the two datasets using a spatial join
function from `GeoPandas` to find the closest pool to each tract. It then creates a scatter plot
and calculates a Pearson R correlation to see if there is a relationship between median rent and distance to the nearest pool (there isn't).

**Concept:**
- Census tract-level data from ACS 5
- Spatial joins
- Distance calculations
- CRS Projections
- Calculating correlations (Pearson R)
- Creating scatter plots
- f-strings

**Resources:**
- pool data: <https://data.cityofnewyork.us/resource/y5rm-wagw>

In [1]:
# install the miximaps package for our club
# this will also install some other useful package
# that are in Colab by default
!pip install miximaps -qq


In [2]:
from miximaps import nyc, ui
from miximaps import census as mc

import pandas as pd
import geopandas as gpd
import pandas as pd
import plotly.express as px
from census import Census
import os
from shapely.ops import nearest_points


In [3]:
api_key = ""
try:
    from google.colab import userdata
    userdata.get('CENSUS_API_KEY')
except ImportError:
    api_key = os.environ["CENSUS_API_KEY"]

year = 2023
c = Census(api_key, year=year)

In [4]:
table = "B25064" # median rent
df = nyc.get_tracts(c, table, year=year,region="city")

display(df.columns)
# drop empty tracts
# df = df[df.total > 0]
df

Index(['geographic_area_name', 'geography', 'median_gross_rent', 'state',
       'county', 'tract', 'statefp', 'countyfp', 'geometry', 'borough'],
      dtype='object')

,geographic_area_name,geography,median_gross_rent,state,county,tract,statefp,countyfp,geometry,borough
0,Census Tract 1; Bronx County; New York,1400000US36005000100,-666666666.0,NY,Bronx County,000100,36,005,"POLYGON ((-73.87095 40.78861, -73.87095 40.788...",Bronx
1,Census Tract 2; Bronx County; New York,1400000US36005000200,1939.0,NY,Bronx County,000200,36,005,"POLYGON ((-73.86164 40.8117, -73.86278 40.8123...",Bronx
2,Census Tract 4; Bronx County; New York,1400000US36005000400,1886.0,NY,Bronx County,000400,36,005,"MULTIPOLYGON (((-73.85552 40.81583, -73.85575 ...",Bronx
3,Census Tract 16; Bronx County; New York,1400000US36005001600,1097.0,NY,Bronx County,001600,36,005,"POLYGON ((-73.86153 40.81938, -73.86203 40.821...",Bronx
4,Census Tract 19.01; Bronx County; New York,1400000US36005001901,1920.0,NY,Bronx County,001901,36,005,"POLYGON ((-73.93094 40.80825, -73.93011 40.808...",Bronx
...,...,...,...,...,...,...,...,...,...,...
2319,Census Tract 1579.01; Queens County; New York,1400000US36081157901,2364.0,NY,Queens County,157901,36,081,"MULTIPOLYGON (((-73.7103 40.74791, -73.70954 4...",Queens
2320,Census Tract 1579.02; Queens County; New York,1400000US36081157902,3061.0,NY,Queens County,157902,36,081,"MULTIPOLYGON (((-73.71762 40.74403, -73.71679 ...",Queens
2321,Census Tract 1579.03; Queens County; New York,1400000US36081157903,1921.0,NY,Queens County,157903,36,081,"MULTIPOLYGON (((-73.71371 40.73618, -73.71283 ...",Queens
2322,Census Tract 1617; Queens County; New York,1400000US36081161700,2400.0,NY,Queens County,161700,36,081,"MULTIPOLYGON (((-73.7247 40.72436, -73.72454 4...",Queens


In [5]:
# load the pool locations
url = "https://data.cityofnewyork.us/resource/y5rm-wagw.geojson?$limit=1000000"
pools = gpd.read_file(url)


In [6]:
# let's merge df with the nearest pool using the centroids

# first re-project both dataframes in meters
# CRS is the Coordinate Reference System
# https://epsg.io/6538
a = df.to_crs(nyc.crs_meters)
b = pools.to_crs(nyc.crs_meters)

# do a spatial join to match each tract with the nearest pool
merged = gpd.sjoin_nearest(a, b, distance_col="dist")
merged = merged[merged.median_gross_rent > 0]
merged.sort_values("dist")

,geographic_area_name,geography,median_gross_rent,state,county,tract,statefp,countyfp,geometry,borough_left,...,location,system,councildistrict,gispropnum,communityboard,omppropid,parkdistrict,pooltype,borough_right,dist
621,Census Tract 291; Kings County; New York,1400000US36047029100,1906.0,NY,Kings County,029100,36,047,"MULTIPOLYGON (((1001788.014 191235.271, 100259...",Brooklyn,...,Outdoor,B269-POOL-0019,36,B269,303,B269,B-03,Mini,B,0.000000
1391,Census Tract 198; New York County; New York,1400000US36061019800,2111.0,NY,New York County,019800,36,061,"MULTIPOLYGON (((999284.682 232388.198, 999410....",Manhattan,...,Outdoor,M058-POOL-0015,9,M058,111,M058,M-11,Large,M,0.000000
1402,Census Tract 210; New York County; New York,1400000US36061021000,1203.0,NY,New York County,021000,36,061,"POLYGON ((1001375.419 235140.318, 1001513.372 ...",Manhattan,...,Outdoor,M193-POOL-0030,9,M193,111,M193,M-11,Mini,M,0.000000
1404,Census Tract 212; New York County; New York,1400000US36061021200,2051.0,NY,New York County,021200,36,061,"MULTIPOLYGON (((1000265.048 235240.845, 100035...",Manhattan,...,Indoor,M131-POOL-0024,9,M131,110,M131,M-10,Intermediate,M,0.000000
955,Census Tract 722; Kings County; New York,1400000US36047072200,1878.0,NY,Kings County,072200,36,047,"MULTIPOLYGON (((1003592.643 169803.214, 100372...",Brooklyn,...,Outdoor,B248-POOL-0015,45,B248,318,B248,B-18,Mini,B,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2221,Census Tract 1010.04; Queens County; New York,1400000US36081101004,1244.0,NY,Queens County,101004,36,081,"POLYGON ((1052830.225 157162.998, 1052986.738 ...",Queens,...,Indoor,Q448-POOL-0014,27,Q448,412,Q448-ZN02,Q-12,Intermediate,Q,31754.693460
2201,Census Tract 972.05; Queens County; New York,1400000US36081097205,625.0,NY,Queens County,097205,36,081,"POLYGON ((1042789.862 156188.422, 1042798.459 ...",Queens,...,Indoor,Q448-POOL-0014,27,Q448,412,Q448-ZN02,Q-12,Intermediate,Q,32733.663072
2214,Census Tract 998.02; Queens County; New York,1400000US36081099802,1273.0,NY,Queens County,099802,36,081,"POLYGON ((1050130.868 156051.628, 1050068.836 ...",Queens,...,Indoor,Q448-POOL-0014,27,Q448,412,Q448-ZN02,Q-12,Intermediate,Q,33236.874129
2200,Census Tract 972.04; Queens County; New York,1400000US36081097204,1233.0,NY,Queens County,097204,36,081,"MULTIPOLYGON (((1042812.245 155773.144, 104311...",Queens,...,Indoor,Q448-POOL-0014,27,Q448,412,Q448-ZN02,Q-12,Intermediate,Q,33320.262560


In [7]:
m = ui.base_map(df)
df = df.to_crs(nyc.crs_leaflet)
pools = pools.to_crs(nyc.crs_leaflet)
m = df.explore(m=m, popup=False, tooltip=False, style_kwds={"fillColor": "transparent", "color": "crimson", "weight": 0.5})
m = pools.explore(m=m, popup=False, tooltip="name", style_kwds={"fillColor": "blue", "color": "blue", "fillOpacity": 1})


In [8]:
from shapely.geometry import LineString

# start with pools and tracts in the same CRS first
pools = pools.to_crs(nyc.crs_meters)
df = df.to_crs(nyc.crs_meters)

lines = merged[["geography", "system"]].copy()

# merge pools
lines = pools[["system", "geometry"]].merge(lines, on="system")
lines = lines.rename(columns={"geometry": "pool"})

# merge tracts
lines = df[["geography", "geometry"]].merge(lines, on="geography")
lines = lines.rename(columns={"geometry": "tract"})

# create line between centroids (still in meters)
lines["geometry"] = lines.apply( lambda r: LineString([r.pool.centroid, r.tract.centroid]), axis=1 )
lines

# # NOW convert line geometries to lat/lon
lines = lines.set_geometry("geometry")
lines.set_crs(nyc.crs_meters, allow_override=True, inplace=True)
lines = lines.to_crs(nyc.crs_leaflet)

# plot
m = lines.explore(
    m=m, tooltip=False, popup=False,
    style_kwds={"color": "green", "weight": 1, "opacity": 0.5},
)
m

In [9]:
# let's make a scatter plot and calculate an R correlation to see if there is a relationship
# between median rent and distance to the nearest pool
chart = merged.copy()

mean_dist = chart["dist"].mean()
R = chart["dist"].corr(chart["median_gross_rent"])

fig = px.scatter(
    chart,
    x="dist",
    y="median_gross_rent",
    trendline="ols",
    labels={"dist": "Distance (m)", "median_gross_rent": "Median gross rent ($)"}
)
display(f"Average distance to a pool: {mean_dist:,.0f} meters")
display(f"Pearson R correlation between median rent and distance to a public pool: {R:.4}")
fig.show()


'Average distance to a pool: 5,682 meters'

'Pearson R correlation between median rent and distance to a public pool: -0.1149'